In [2]:
import torch
import torch.nn as nn

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. LOAD THE DIGIT DATA
# ============================================================

digits = load_digits()

X = digits.data
y = digits.target

print("X shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# 2. SPLIT INTO TRAINING AND TEST DATA
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# 3. SCALE THE DATA
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ============================================================
# 4. CONVERT NUMPY → PYTORCH TENSORS
# ============================================================

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


# ============================================================
# 5. CREATE THE NEURAL NETWORK
# ============================================================

model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

print("\nModel:")
print(model)


# ============================================================
# 6. LOSS FUNCTION
# ============================================================

loss_fn = nn.CrossEntropyLoss()


# ============================================================
# 7. OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)


# ============================================================
# 8. TRAIN THE NETWORK
# ============================================================

for epoch in range(100):

    # ---- Forward ----
    predictions = model(X_train)

    # ---- Measure mistake ----
    loss = loss_fn(predictions, y_train)

    # ---- Forget old gradients ----
    optimizer.zero_grad()

    # ---- Calculate new gradients ----
    loss.backward()

    # ---- Update weights ----
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1:3d} | "
            f"Loss: {loss.item():.4f}"
        )


# ============================================================
# 9. TEST THE MODEL
# ============================================================

with torch.no_grad():

    test_output = model(X_test)

    predictions = test_output.argmax(dim=1)

    accuracy = (predictions == y_test).float().mean()

print("\nTest accuracy:", accuracy.item())


# ============================================================
# 10. LOOK AT SOME PREDICTIONS
# ============================================================

print("\nFirst 20 predictions:")

for i in range(20):

    print(
        "Actual:",
        y_test[i].item(),
        "| Predicted:",
        predictions[i].item()
    )

X shape: (1797, 64)
y shape: (1797,)

Model:
Sequential(
  (0): Linear(in_features=64, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=10, bias=True)
)
Epoch  10 | Loss: 0.9434
Epoch  20 | Loss: 0.2987
Epoch  30 | Loss: 0.1268
Epoch  40 | Loss: 0.0679
Epoch  50 | Loss: 0.0405
Epoch  60 | Loss: 0.0265
Epoch  70 | Loss: 0.0185
Epoch  80 | Loss: 0.0139
Epoch  90 | Loss: 0.0109
Epoch 100 | Loss: 0.0088

Test accuracy: 0.9722222089767456

First 20 predictions:
Actual: 5 | Predicted: 5
Actual: 2 | Predicted: 2
Actual: 8 | Predicted: 8
Actual: 1 | Predicted: 1
Actual: 7 | Predicted: 7
Actual: 2 | Predicted: 2
Actual: 6 | Predicted: 6
Actual: 2 | Predicted: 2
Actual: 6 | Predicted: 6
Actual: 5 | Predicted: 5
Actual: 0 | Predicted: 0
Actual: 5 | Predicted: 5
Actual: 9 | Predicted: 9
Actual: 3 | Predicted: 3
Actual: 4 | Predicted: 4
Actual: 4 | Predicted: 4
Actual: 2 | Predicted: 2
Actual: 4 | Predicted: 4
Actual: 9 | Predicted: 9
Actual: 9 | Predicted: 9


In [3]:
print("\nWrong predictions:")

wrong = torch.where(predictions != y_test)[0]

print("Number wrong:", len(wrong))

for i in wrong[:10]:

    print(
        "Actual:",
        y_test[i].item(),
        "| Predicted:",
        predictions[i].item()
    )


Wrong predictions:
Number wrong: 10
Actual: 8 | Predicted: 1
Actual: 8 | Predicted: 7
Actual: 1 | Predicted: 4
Actual: 4 | Predicted: 9
Actual: 8 | Predicted: 1
Actual: 0 | Predicted: 4
Actual: 3 | Predicted: 5
Actual: 1 | Predicted: 4
Actual: 8 | Predicted: 1
Actual: 8 | Predicted: 3
